# Environment Setup

## Mount Drive + define project paths

In [1]:
from google.colab import drive
# This imports the Drive utility that lets Colab access your Google Drive.
# It is required so we can save your dataset, runs, and deliverables to your course folder.

drive.mount('/content/drive')
# This mounts your Drive into the Colab filesystem at /content/drive.
# After mounting, we can read and write files to your "SA 2 - Dequiñon_Pasamonte" folder.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Install Requirements

In [4]:
# =========================
# REQUIREMENTS CELL (FIXED FOR GOOGLE COLAB)
# =========================

!pip -q install --upgrade ultralytics roboflow pyyaml matplotlib
# This installs Ultralytics (YOLO training), Roboflow (dataset download), and plotting/YAML helpers.
# It avoids upgrading pandas, because Colab has pinned dependencies that break with pandas 3.x.

!pip -q install "pandas==2.2.2"
# This forces the exact pandas version required by Google Colab.
# It prevents dependency conflicts with preinstalled Colab packages that require pandas < 3.0.

import os
# This imports OS utilities used later for file/folder operations and runtime control.
# It is also used to help verify environment details when debugging.

import glob
# This imports file pattern matching utilities for locating files like data.yaml and confusion_matrix.png.
# It helps you reliably find artifacts even if folder structures vary.

import yaml
# This imports a YAML parser used to read and edit your data.yaml file.
# A correct data.yaml is required so YOLO can find train/val images and class names.

import shutil
# This imports file copying utilities used to move the dataset and outputs into Google Drive.
# It ensures your dataset and runs persist after the Colab session ends.

import pandas as pd
# This imports Pandas for the required formatted tables (hyperparameters + metrics).
# Pandas is also used to read results.csv and display final epoch outputs.

from IPython.display import Image, display
# This imports notebook display helpers for showing confusion matrix images inside the notebook.
# Displaying confusion_matrix.png is one of the required deliverables.

import ultralytics
# This imports Ultralytics to confirm installation and version.
# Printing the version helps verify YOLO tooling is available.

from ultralytics import YOLO
# This imports the YOLO class used to train and validate models.
# It provides the .train() and .val() methods required by your activity.

print("✅ pandas:", pd.__version__)
# This prints the pandas version currently active in the runtime.
# It should be 2.2.2 to match Colab compatibility.

print("✅ ultralytics:", ultralytics.__version__)
# This prints the installed Ultralytics version.
# Use this output if you need to report your environment in the submission.

✅ pandas: 2.2.2
✅ ultralytics: 8.4.19


## Define Folder Paths

In [6]:
COURSE_ROOT = "/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte"
# This sets the exact path to your course folder in Google Drive.
# It matches your stated folder name so outputs land in the correct location.

DRIVE_DATASET_DIR = os.path.join(COURSE_ROOT, "dataset")
# This defines the dataset folder inside your course folder.
# The activity requires your dataset to be stored and referenced consistently.

DRIVE_RUNS_DIR = os.path.join(COURSE_ROOT, "runs")
# This defines where training outputs (runs) will be copied for your deliverables.
# Keeping runs in Drive makes it easy to share cloud links and preserve results.

os.makedirs(DRIVE_DATASET_DIR, exist_ok=True)
# This creates the dataset folder if it does not already exist.
# It prevents errors when we copy the Roboflow export to Drive.

os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)
# This creates the runs folder if it does not already exist.
# It ensures all artifacts can be saved for later download and sharing.

print("COURSE_ROOT:", COURSE_ROOT)
# This prints your root path so you can visually verify it is correct.
# A correct path avoids broken file references when training starts.

print("DRIVE_DATASET_DIR:", DRIVE_DATASET_DIR)
# This prints where the dataset will be placed in Drive.
# This location will be used in the final data.yaml path fix.

print("DRIVE_RUNS_DIR:", DRIVE_RUNS_DIR)
# This prints where we will copy YOLO run outputs after training.
# This is the folder you will share as a cloud link for submission.

COURSE_ROOT: /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte
DRIVE_DATASET_DIR: /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset
DRIVE_RUNS_DIR: /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/runs


## Dataset

In [7]:
from getpass import getpass
# This imports a secure input method that hides typed text in the notebook.
# It prevents your API key from being printed or stored in outputs.

ROBOFLOW_API_KEY = getpass("Paste your Roboflow API key (input hidden): ")
# This collects your Roboflow API key without displaying it on screen.
# Hiding the key helps you comply with good security practice for submissions.

from roboflow import Roboflow
# This imports the Roboflow SDK used to download your dataset export.
# It enables workspace/project/version access and format exports.

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# This authenticates your Roboflow session using the key you provided.
# Successful auth is required to download the dataset programmatically.

project = rf.workspace("hezekiahs-workspace").project("my-first-project-lvodo")
# This selects the exact workspace and project you stated.
# The dataset content and classes come from this project definition.

version = project.version(1)
# This selects version 1 of the dataset, matching your teammate’s code.
# Versioning matters because splits and labels can change per version.

dataset = version.download("yolo26")
# This downloads the dataset exported in YOLO26 format (as requested).
# The export typically includes images/labels and a data.yaml file.

Paste your Roboflow API key (input hidden): ··········
loading Roboflow workspace...
loading Roboflow project...


In [8]:
rf_dataset_dir = dataset.location
# This reads the local Colab path where Roboflow downloaded the dataset.
# We must know the exact source folder before copying to Drive.

print("Roboflow downloaded dataset to:", rf_dataset_dir)
# This prints the download folder so you can confirm it exists.
# If the folder does not exist, training would fail due to missing data.

if os.path.exists(DRIVE_DATASET_DIR):
    # This checks if the destination dataset folder exists in Drive.
    # The folder was created earlier, but this ensures safe behavior anyway.
    pass
# This does nothing but keeps structure explicit for readability.
# It also avoids accidental deletion of your Drive folder.

shutil.copytree(rf_dataset_dir, DRIVE_DATASET_DIR, dirs_exist_ok=True)
# This copies the entire Roboflow-exported YOLO dataset into your Drive dataset folder.
# Copying into Drive ensures persistence after the Colab session ends.

print("✅ Dataset copied to Drive:", DRIVE_DATASET_DIR)
# This confirms the dataset is now stored in the required Drive folder.
# Your subsequent training will reference this Drive-stored dataset.

Roboflow downloaded dataset to: /content/My-First-Project-1
✅ Dataset copied to Drive: /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset


## Data.yaml File Creation

In [9]:
yaml_candidates = glob.glob(os.path.join(DRIVE_DATASET_DIR, "**", "data.yaml"), recursive=True)
# This searches your Drive dataset folder for the YOLO data.yaml file.
# YOLO training depends on this file to locate training and validation images.

print("Found data.yaml candidates:", yaml_candidates)
# This prints any detected data.yaml paths.
# Seeing at least one result confirms your export includes the required config.

DATA_YAML = yaml_candidates[0]
# This selects the first matched data.yaml file as the active config.
# Roboflow exports usually contain a single correct data.yaml in the root.

with open(DATA_YAML, "r") as f:
    # This opens the data.yaml file for reading.
    # We need to inspect it to ensure train/val paths are correct.
    data_cfg = yaml.safe_load(f)
# This parses YAML into a Python dictionary for editing.
# Editing in-memory is safer than manually modifying the file.

print("Original data.yaml content:", data_cfg)
# This prints the loaded YAML config so you can verify keys like train/val/names.
# This helps confirm the dataset is set up for detection and not classification.

def _abs_if_relative(p):
    # This defines a helper to convert relative paths into absolute Drive paths.
    # It avoids broken training paths when Roboflow uses relative references.
    if p is None:
        return None
    # This handles missing values safely by returning None.
    # It prevents crashing if a key is absent in the YAML.
    if os.path.isabs(str(p)):
        return str(p)
    # This preserves already-absolute paths without modification.
    # It avoids accidentally corrupting correct absolute paths.
    return os.path.join(DRIVE_DATASET_DIR, str(p))
# This joins relative paths against your Drive dataset folder.
# This ensures YOLO always reads images/labels from Drive reliably.

if "path" in data_cfg and data_cfg["path"] is not None:
    # This checks if the YAML uses a root "path" key.
    # Some YOLO configs use it to resolve train/val relative locations.
    data_cfg["path"] = DRIVE_DATASET_DIR
# This forces the dataset root to be the Drive dataset folder.
# This makes relative train/val paths resolve inside your Drive directory.

data_cfg["train"] = _abs_if_relative(data_cfg.get("train"))
# This updates the train path to an absolute Drive path.
# Absolute paths are more reliable in Colab than relative paths.

data_cfg["val"] = _abs_if_relative(data_cfg.get("val"))
# This updates the val path to an absolute Drive path.
# Validation must work to produce the confusion matrix and metrics.

with open(DATA_YAML, "w") as f:
    # This opens the YAML file for writing back the corrected paths.
    # Updating the file ensures Ultralytics uses the correct locations.
    yaml.safe_dump(data_cfg, f, sort_keys=False)
# This writes the corrected YAML while preserving key order.
# A correct YAML prevents “images not found” training errors.

print("✅ Fixed data.yaml saved at:", DATA_YAML)
# This confirms the YAML rewrite succeeded and prints the saved location.
# You will use DATA_YAML as the data parameter in training.

print("Updated data.yaml content:", data_cfg)
# This prints the updated YAML so you can confirm train/val are correct.
# Correct train/val paths are required for successful training and evaluation.

Found data.yaml candidates: ['/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/data.yaml']
Original data.yaml content: {'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 6, 'names': ['bus', 'car', 'jeepney', 'motorcycle', 'tricycle', 'truck'], 'roboflow': {'workspace': 'hezekiahs-workspace', 'project': 'my-first-project-lvodo', 'version': 1, 'license': 'Private', 'url': 'https://app.roboflow.com/hezekiahs-workspace/my-first-project-lvodo/1'}}
✅ Fixed data.yaml saved at: /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/data.yaml
Updated data.yaml content: {'train': '/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/../train/images', 'val': '/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/../valid/images', 'test': '../test/images', 'nc': 6, 'names': ['bus', 'car', 'jeepney', 'motorcycle', 'tricycle', 'truck'], 'roboflow': {'workspace': 'hezekiahs-workspace', 'project': 'my-first-project-lvodo', 'version': 1, 'license': '

## Dataset preparation

In [14]:
# =========================
# CELL A — Inspect dataset structure
# =========================

import os
# This imports OS utilities for filesystem checks.
# We will use it to verify required folders exist before splitting.

DATASET_ROOT = "/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset"
# This sets the dataset root folder exactly as you described in Drive.
# All images and labels should be inside this directory.

train_images = os.path.join(DATASET_ROOT, "train", "images")
# This constructs the expected path for training images in YOLO format.
# Your earlier output confirms this folder exists.

train_labels = os.path.join(DATASET_ROOT, "train", "labels")
# This constructs the expected path for training labels in YOLO format.
# YOLO label files must match image filenames for training to work.

print("train_images exists:", os.path.exists(train_images), train_images)
# This prints whether the training images folder exists.
# If False, your export is not in standard YOLO folder structure.

print("train_labels exists:", os.path.exists(train_labels), train_labels)
# This prints whether the training labels folder exists.
# If False, you may have an export with a different label folder name.

print("Sample train images:", sorted(os.listdir(train_images))[:5] if os.path.exists(train_images) else "N/A")
# This prints a few training image filenames if the folder exists.
# Seeing filenames confirms the dataset is readable by Colab.

print("Sample train labels:", sorted(os.listdir(train_labels))[:5] if os.path.exists(train_labels) else "N/A")
# This prints a few training label filenames if the folder exists.
# Seeing .txt files confirms YOLO annotations are present.

train_images exists: True /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/train/images
train_labels exists: True /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/train/labels
Sample train images: ['clear_day_f000000_jpg.rf.d5f6a64e4a31fa2d4c08bd5f1fb66852.jpg', 'clear_day_f000030_jpg.rf.ddab3a111355a3d0e880b5fd3e3fae43.jpg', 'clear_day_f000060_jpg.rf.ed931f15fe27c6846563453ac222d08b.jpg', 'clear_day_f000090_jpg.rf.82a169b44e369e916ea8b987ed78781a.jpg', 'clear_day_f000120_jpg.rf.0afaa7d926c92ffca181f3d47f4f48d6.jpg']
Sample train labels: ['clear_day_f000000_jpg.rf.d5f6a64e4a31fa2d4c08bd5f1fb66852.txt', 'clear_day_f000030_jpg.rf.ddab3a111355a3d0e880b5fd3e3fae43.txt', 'clear_day_f000060_jpg.rf.ed931f15fe27c6846563453ac222d08b.txt', 'clear_day_f000090_jpg.rf.82a169b44e369e916ea8b987ed78781a.txt', 'clear_day_f000120_jpg.rf.0afaa7d926c92ffca181f3d47f4f48d6.txt']


In [15]:
# =========================
# CELL B — Create val/ split folders
# =========================

import shutil
# This imports file copying tools used to move images/labels into val folders.
# We will use it to build a proper validation set required by your activity.

import random
# This imports random sampling utilities for selecting validation images.
# We will use it to create an unbiased val split.

val_images = os.path.join(DATASET_ROOT, "val", "images")
# This defines the new validation images folder we will create.
# Ultralytics accepts "val" as the standard validation folder name.

val_labels = os.path.join(DATASET_ROOT, "val", "labels")
# This defines the new validation labels folder we will create.
# Every val image must have a corresponding YOLO label text file.

os.makedirs(val_images, exist_ok=True)
# This creates the val/images folder if it doesn’t exist.
# The folder is needed so Ultralytics can load validation images.

os.makedirs(val_labels, exist_ok=True)
# This creates the val/labels folder if it doesn’t exist.
# The folder is needed so Ultralytics can load validation labels.

In [16]:
# =========================
# CELL C — Split train into train+val (80/20) by MOVING files
# =========================

img_files = sorted([f for f in os.listdir(train_images) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))])
# This collects all image filenames from train/images.
# We filter to common image extensions so only valid images are split.

random.seed(42)
# This fixes the randomness so your split is reproducible.
# Reproducibility helps when you re-run training and want consistent results.

val_count = max(1, int(0.2 * len(img_files)))
# This computes how many images go to validation based on 20% of the dataset.
# We ensure at least one validation image so validation can run.

val_imgs = set(random.sample(img_files, val_count))
# This randomly selects the images that will move into the validation set.
# Using a set makes membership checks fast and clean.

missing_labels = 0
# This initializes a counter for label files that are missing.
# Missing labels indicate dataset inconsistencies that can harm training.

moved = 0
# This initializes a counter for how many image-label pairs were moved.
# Counting moved pairs verifies that the split actually happened.

for img in img_files:
    # This iterates through every training image to decide if it goes to val.
    # We move only the selected subset so training keeps the remaining images.

    if img not in val_imgs:
        # This skips images that were not selected for validation.
        # Skipping keeps most images in the training set.

        continue
        # This continues to the next image filename.
        # It ensures only selected images are processed for moving.

    src_img = os.path.join(train_images, img)
    # This builds the source path for the image file in train/images.
    # We must know the exact file location before moving it.

    dst_img = os.path.join(val_images, img)
    # This builds the destination path for the image file in val/images.
    # The filename stays the same so label matching remains consistent.

    label_name = os.path.splitext(img)[0] + ".txt"
    # This converts the image filename into the expected YOLO label filename.
    # YOLO labels use the same base name with a .txt extension.

    src_lbl = os.path.join(train_labels, label_name)
    # This builds the source path for the label file in train/labels.
    # Each image must have a matching label file for detection training.

    dst_lbl = os.path.join(val_labels, label_name)
    # This builds the destination path for the label file in val/labels.
    # Keeping labels with images is required for correct validation.

    shutil.move(src_img, dst_img)
    # This moves the image file from train/images to val/images.
    # Moving avoids duplicating data and keeps the dataset size consistent.

    if os.path.exists(src_lbl):
        # This checks if the corresponding label file exists before moving it.
        # Missing labels must be tracked because they break supervised learning.

        shutil.move(src_lbl, dst_lbl)
        # This moves the label file from train/labels to val/labels.
        # Matching image-label pairs ensures validation metrics are meaningful.
    else:
        missing_labels += 1
        # This increments the missing label counter if the label file is not found.
        # You may need to fix these cases if the count is large.

    moved += 1
    # This increments the moved counter after processing one validation image.
    # It helps confirm the number of moved images matches your target split.

print("Total images found:", len(img_files))
# This prints the number of images originally in train/images.
# It helps you verify dataset size before and after the split.

print("Validation images moved:", moved)
# This prints how many images were moved into val/images.
# It should match val_count unless some images were missing or filtered.

print("Missing labels among moved:", missing_labels)
# This prints how many moved images had no label file.
# If this is not near zero, your dataset annotations may be incomplete.

print("Train images now:", len(os.listdir(train_images)))
# This prints the remaining number of training images after moving validation images.
# A decreased count confirms that files were actually moved.

print("Val images now:", len(os.listdir(val_images)))
# This prints the number of validation images after the move.
# A non-zero count is required for validation, confusion matrix, and metrics.

Total images found: 1396
Validation images moved: 279
Missing labels among moved: 0
Train images now: 1117
Val images now: 279


In [17]:
# =========================
# CELL D — Rewrite data.yaml to use train and val folders (absolute paths)
# =========================

import glob
# This imports glob for finding data.yaml automatically.
# It prevents manual path mistakes when your export structure changes.

import yaml
# This imports YAML utilities for reading and writing data.yaml.
# Ultralytics relies on correct YAML train/val paths.

yaml_candidates = glob.glob(os.path.join(DATASET_ROOT, "**", "data.yaml"), recursive=True)
# This searches for the data.yaml file in your dataset folder.
# Roboflow exports should include exactly one data.yaml.

DATA_YAML = yaml_candidates[0]
# This selects the first found data.yaml path.
# Using the file in your dataset ensures classes/names remain correct.

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)
# This loads the YAML into a Python dictionary so we can update paths.
# We keep existing class names and only fix the split paths.

data_cfg["train"] = train_images
# This sets the training path to your Drive train/images folder.
# It ensures Ultralytics loads training images from the correct place.

data_cfg["val"] = val_images
# This sets the validation path to your new Drive val/images folder.
# It enables Ultralytics to compute confusion matrix and validation metrics.

if "path" in data_cfg:
    data_cfg.pop("path", None)
# This removes a root path key if it exists.
# Removing it prevents Ultralytics from rebuilding incorrect absolute paths.

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
# This writes the corrected YAML back to Drive.
# Saving the file makes the fix permanent for training and validation.

print("✅ Fixed data.yaml:", DATA_YAML)
# This prints the YAML file location you will use in training.
# It confirms you are editing the correct file in Drive.

print("✅ YAML content:", data_cfg)
# This prints the final YAML dictionary so you can verify train and val paths.
# Correct paths are required to avoid the 'images not found' error.

✅ Fixed data.yaml: /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/data.yaml
✅ YAML content: {'train': '/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/train/images', 'val': '/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/val/images', 'test': '../test/images', 'nc': 6, 'names': ['bus', 'car', 'jeepney', 'motorcycle', 'tricycle', 'truck'], 'roboflow': {'workspace': 'hezekiahs-workspace', 'project': 'my-first-project-lvodo', 'version': 1, 'license': 'Private', 'url': 'https://app.roboflow.com/hezekiahs-workspace/my-first-project-lvodo/1'}}


# Yolo Implementation

## Parameters Setup

In [18]:
BASE_MODEL = "yolo26n.pt"
# This sets the YOLOv26 detection model checkpoint to initialize each run.
# Using the same base model ensures fair comparison across hyperparameter changes.

RUNS = [
    {"run_name": "model1_e25_adamw_b4_lr1e-2",   "epochs": 25, "imgsz": 640, "optimizer": "AdamW", "batch": 4,  "lr0": 0.01},
    # This run uses epochs=25, optimizer=AdamW, batch=4, and lr0=0.01 exactly as required.
    # It serves as one corner of your hyperparameter grid for comparison.

    {"run_name": "model2_e30_sgd_b20_lr1e-3",    "epochs": 30, "imgsz": 640, "optimizer": "SGD",   "batch": 20, "lr0": 0.001},
    # This run uses epochs=30, optimizer=SGD, batch=20, and lr0=0.001 exactly as required.
    # It provides a contrasting optimizer and batch setting to observe changes in precision/recall.

    {"run_name": "model3_e40_auto_bauto_lr1e-4", "epochs": 40, "imgsz": 640, "optimizer": "auto",  "batch": -1, "lr0": 0.0001},
    # This run uses epochs=40, optimizer=auto, batch=-1 (auto batching), and lr0=0.0001 exactly as required.
    # It tests longer training with automatic optimizer selection and automatic batching behavior.
]
# This list defines all three training configurations for the activity.
# Using a list lets you loop cleanly and also build the required hyperparameter table.

hp_table = pd.DataFrame(RUNS)
# This converts your run definitions into a Pandas table for the deliverable.
# It ensures your hyperparameter settings are displayed in a formatted way.

hp_table
# This displays the hyperparameter table directly in the notebook output.
# A visible table satisfies the deliverable for hyperparameter documentation.

,run_name,epochs,imgsz,optimizer,batch,lr0
0,model1_e25_adamw_b4_lr1e-2,25,640,AdamW,4,0.0100
1,model2_e30_sgd_b20_lr1e-3,30,640,SGD,20,0.0010
2,model3_e40_auto_bauto_lr1e-4,40,640,auto,-1,0.0001


## Training

In [19]:
# =========================
# DATA.YAML PATH FIX CELL (RUN THIS BEFORE TRAINING)
# =========================

import os
# This imports OS path utilities to build and verify folder locations.
# We need it to find the real train/val image directories inside your Drive dataset folder.

import glob
# This imports glob for searching folder patterns like */train/images and */valid/images.
# Roboflow exports can vary, so pattern search makes the fix robust.

import yaml
# This imports YAML utilities to read and rewrite data.yaml cleanly.
# Ultralytics relies on correct YAML paths to locate images and labels.

# 1) Confirm the dataset root in Drive
DATASET_ROOT = "/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset"
# This sets the dataset root to the folder you said exists in Drive.
# All train/val image folders must be inside this directory.

print("DATASET_ROOT:", DATASET_ROOT)
# This prints the dataset root so you can verify the path is correct.
# A wrong root path will cause all later searches to fail.

# 2) Locate the actual train images folder
train_candidates = glob.glob(os.path.join(DATASET_ROOT, "**", "train", "images"), recursive=True)
# This searches for a YOLO-style training images directory.
# Roboflow YOLO exports typically use train/images.

# 3) Locate the actual validation images folder (valid/images or val/images)
valid_candidates = glob.glob(os.path.join(DATASET_ROOT, "**", "valid", "images"), recursive=True)
# This searches for Roboflow's common validation folder name "valid/images".
# Many Roboflow exports use 'valid' instead of 'val'.

val_candidates = glob.glob(os.path.join(DATASET_ROOT, "**", "val", "images"), recursive=True)
# This searches for the alternative validation folder name "val/images".
# Some exports follow the standard YOLO naming convention 'val'.

print("train_candidates:", train_candidates)
# This prints found training image directories so you can see what was detected.
# At least one match must exist to proceed.

print("valid_candidates:", valid_candidates)
# This prints found 'valid/images' directories if present.
# If empty, we will fall back to 'val/images'.

print("val_candidates:", val_candidates)
# This prints found 'val/images' directories if present.
# We use this if 'valid/images' is not available.

if not train_candidates:
    raise FileNotFoundError("❌ Could not find train/images inside your Drive dataset folder.")
# This stops execution if no training images folder is found.
# Training cannot run without a valid training path.

TRAIN_IMAGES = train_candidates[0]
# This selects the first discovered train/images folder.
# Roboflow usually provides only one train/images path.

if valid_candidates:
    VAL_IMAGES = valid_candidates[0]
else:
    if not val_candidates:
        raise FileNotFoundError("❌ Could not find valid/images or val/images inside your Drive dataset folder.")
    VAL_IMAGES = val_candidates[0]
# This picks valid/images if it exists, otherwise val/images.
# The goal is to ensure Ultralytics has a real validation images folder.

print("✅ TRAIN_IMAGES:", TRAIN_IMAGES)
# This prints the final resolved training images directory.
# You will confirm it is inside your Drive dataset folder.

print("✅ VAL_IMAGES:", VAL_IMAGES)
# This prints the final resolved validation images directory.
# You will confirm it is inside your Drive dataset folder.

# 4) Locate data.yaml inside the dataset root
yaml_candidates = glob.glob(os.path.join(DATASET_ROOT, "**", "data.yaml"), recursive=True)
# This searches for data.yaml within your dataset folder.
# Roboflow exports should include a data.yaml near the dataset root.

if not yaml_candidates:
    raise FileNotFoundError("❌ data.yaml not found inside your Drive dataset folder.")
# This stops execution if data.yaml is missing.
# YOLO training requires this config file.

DATA_YAML = yaml_candidates[0]
# This selects the first discovered data.yaml file.
# In typical exports, there is only one.

print("✅ DATA_YAML:", DATA_YAML)
# This prints the chosen YAML path for verification.
# You will pass this exact path into model.train().

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)
# This reads the YAML into a dictionary so we can edit it.
# Editing ensures paths point to real folders in Drive.

data_cfg["train"] = TRAIN_IMAGES
# This sets the YAML train path to the correct absolute Drive location.
# It eliminates broken relative paths like ../train/images.

data_cfg["val"] = VAL_IMAGES
# This sets the YAML val path to the correct absolute Drive location.
# It fixes the current bug where Ultralytics looks in the wrong folder.

# Remove "path" to avoid it overriding train/val in unexpected ways
if "path" in data_cfg:
    data_cfg.pop("path", None)
# This removes the root path key if present.
# In some configs, 'path' can cause Ultralytics to rebuild wrong absolute paths.

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
# This writes the corrected YAML back to disk.
# Saving ensures Ultralytics uses the fixed paths during training.

print("\n✅ FIXED data.yaml content:\n", data_cfg)
# This prints the final YAML dictionary so you can confirm correctness.
# If train/val point to real Drive paths, training should start successfully.

DATASET_ROOT: /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset
train_candidates: ['/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/train/images']
valid_candidates: []
val_candidates: ['/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/val/images']
✅ TRAIN_IMAGES: /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/train/images
✅ VAL_IMAGES: /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/val/images
✅ DATA_YAML: /content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/data.yaml

✅ FIXED data.yaml content:
 {'train': '/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/train/images', 'val': '/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/val/images', 'test': '../test/images', 'nc': 6, 'names': ['bus', 'car', 'jeepney', 'motorcycle', 'tricycle', 'truck'], 'roboflow': {'workspace': 'hezekiahs-workspace', 'project': 'my-first-project-lvodo', 'version': 1, 'license': 'Private', 'url': 'https://app.roboflow.com/hezekiahs-workspace/my-first-

In [22]:
TRAIN_PROJECT = "/content/runs"
# This sets the local output directory where Ultralytics will create run folders.
# Writing locally is faster than writing directly to Google Drive.

trained_weight_paths = {}
# This initializes a dictionary to store best.pt locations per run.
# We will use these paths for validation and metrics extraction later.

for cfg in RUNS:
    # This loops through each training configuration to run three experiments.
    # The loop ensures consistent training procedure across runs.

    model = YOLO(BASE_MODEL)
    # This initializes a fresh YOLO model for each run.
    # Fresh models avoid leaking state between experiments.

    results = model.train(
        data=DATA_YAML,
        # This points training to your corrected YAML file.
        # Correct paths are required so images and labels load correctly.

        epochs=cfg["epochs"],
        # This sets epochs to 25, 30, and 40 as required.
        # We do not change epochs because the course requires these exact values.

        imgsz=cfg["imgsz"],
        # This keeps image size at 640 as required.
        # Image size is not reduced because your instructions fix it at 640.

        optimizer=cfg["optimizer"],
        # This keeps optimizers as AdamW, SGD, and auto as required.
        # We do not change optimizers because it is part of your experimental design.

        batch=cfg["batch"],
        # This keeps batch sizes as 4, 20, and -1 as required.
        # We do not change batch sizes because it is part of your experimental design.

        lr0=cfg["lr0"],
        # This keeps learning rates as 0.01, 0.001, and 0.0001 as required.
        # We do not change learning rates because it is part of your experimental design.

        project=TRAIN_PROJECT,
        # This writes all outputs locally first for speed.
        # Local disk writes are faster than Google Drive writes.

        name=cfg["run_name"],
        # This names each run folder uniquely.
        # Unique run names keep outputs from overwriting each other.

        cache=True,
        # This caches images to reduce disk reads each epoch.
        # Caching usually speeds training significantly in Colab.

        workers=2,
        # This reduces dataloader worker overhead in Colab.
        # Too many workers can slow down or stall on shared environments.

        plots=False,
        # This disables extra plotting during training.
        # Disabling plots reduces overhead and speeds epochs.

        save_period=0,
        # This prevents saving periodic checkpoints every N epochs.
        # It reduces I/O overhead while still saving best.pt and last.pt.

        verbose=True
        # This keeps training logs visible so you can screenshot final epoch output.
        # Logs are required for your deliverable evidence.
    )

    best_pt = os.path.join(TRAIN_PROJECT, "detect", cfg["run_name"], "weights", "best.pt")
    # This constructs the expected path to best.pt for the run.
    # best.pt is used for final validation and metric extraction.

    trained_weight_paths[cfg["run_name"]] = best_pt
    # This stores the best weights path for later evaluation.
    # Storing paths makes the validation loop straightforward.

print("✅ Best weight paths:", trained_weight_paths)
# This prints the best.pt locations to confirm training produced weights.
# Missing best.pt indicates a failed run or wrong output directory.

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/SA 2 - Dequiñon_Pasamonte/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=model1_e25_adamw_b4_lr1e-26, nbs=64, nms=False, opset=None, optimize=False, opt

## Confusion Matrices

In [23]:
confusion_paths = {}
# This initializes a dictionary to store confusion matrix image paths per run.
# We will use it for copying artifacts to Drive and for your report discussion.

for cfg in RUNS:
    # This loops through each run to locate and display its confusion matrix.
    # Each run should have its own confusion_matrix.png in the run folder.

    cm_path = os.path.join(TRAIN_PROJECT, "detect", cfg["run_name"], "confusion_matrix.png")
    # This builds the expected file path for Ultralytics confusion matrix output.
    # The confusion matrix is required for your evaluation deliverable.

    confusion_paths[cfg["run_name"]] = cm_path
    # This records the confusion matrix path for later copying and referencing.
    # Storing paths keeps your notebook organized and reproducible.

    print("Confusion matrix path:", cm_path)
    # This prints the path so you can confirm the file exists in the expected location.
    # A missing file typically means training did not complete or outputs differ.

    if os.path.exists(cm_path):
        # This checks whether the confusion matrix file exists before displaying it.
        # The check prevents the notebook from crashing if the file is missing.
        display(Image(filename=cm_path, width=900))
    else:
        # This branch handles the case where the image does not exist.
        # It lets you still proceed while clearly seeing which run lacks the artifact.
        print("❌ confusion_matrix.png not found for", cfg["run_name"])
# This ends the loop after attempting to display all confusion matrices.
# Having the images shown in-notebook satisfies the plotting deliverable.

Confusion matrix path: /content/runs/detect/model1_e25_adamw_b4_lr1e-2/confusion_matrix.png
❌ confusion_matrix.png not found for model1_e25_adamw_b4_lr1e-2
Confusion matrix path: /content/runs/detect/model2_e30_sgd_b20_lr1e-3/confusion_matrix.png
❌ confusion_matrix.png not found for model2_e30_sgd_b20_lr1e-3
Confusion matrix path: /content/runs/detect/model3_e40_auto_bauto_lr1e-4/confusion_matrix.png
❌ confusion_matrix.png not found for model3_e40_auto_bauto_lr1e-4


## Validation

In [ ]:
# =========================
# CELL 10 — Validate each best.pt and extract metrics (mAP50, Precision, Recall, F1)
# =========================

metric_rows = []
# This creates a list that will store one metrics record per run.
# The list will become a Pandas table required by the activity deliverables.

for cfg in RUNS:
    # This loops over each run to validate the final/best model.
    # Validation produces the final performance statistics used for your summary table.

    run_name = cfg["run_name"]
    # This stores the current run name for cleaner code.
    # It prevents repeated dictionary lookups and improves readability.

    best_pt = trained_weight_paths[run_name]
    # This retrieves the best weights file path for the run.
    # Using best.pt is standard to evaluate the best checkpoint, not last epoch.

    model = YOLO(best_pt)
    # This loads the trained model weights for evaluation.
    # Loading the weights is required before calling model.val().

    v = model.val(data=DATA_YAML, imgsz=640)
    # This runs validation on the dataset specified by your YAML and image size 640.
    # Validation computes mAP, precision, recall, and other diagnostic metrics.

    p = None
    # This initializes precision to None in case the field is unavailable in your version.
    # Defensive initialization prevents errors and allows graceful reporting.

    r = None
    # This initializes recall to None in case the field is unavailable in your version.
    # Defensive initialization prevents errors and allows graceful reporting.

    map50 = None
    # This initializes mAP50 to None in case the field is unavailable in your version.
    # We then attempt extraction using common Ultralytics attributes.

    if hasattr(v, "box") and hasattr(v.box, "map50"):
        # This checks a common Ultralytics location for mAP50 on detection tasks.
        # Using hasattr prevents attribute errors if the structure differs.
        map50 = float(v.box.map50)
    # This stores the mAP50 value as a float when available.
    # mAP50 acts as your accuracy proxy in the required table.

    if hasattr(v, "box") and hasattr(v.box, "mp"):
        # This checks a common Ultralytics location for mean precision.
        # Precision is needed for the deliverable performance table.
        p = float(v.box.mp)
    # This stores precision as a float when available.
    # Precision reflects how many predicted positives are correct.

    if hasattr(v, "box") and hasattr(v.box, "mr"):
        # This checks a common Ultralytics location for mean recall.
        # Recall is needed for the deliverable performance table.
        r = float(v.box.mr)
    # This stores recall as a float when available.
    # Recall reflects how many real positives were successfully detected.

    f1 = None
    # This initializes F1 to None in case precision/recall are missing.
    # We compute F1 only when precision and recall are valid numbers.

    if p is not None and r is not None and (p + r) > 0:
        # This ensures precision and recall exist and the denominator is non-zero.
        # It prevents division errors and guarantees a valid F1 computation.
        f1 = 2 * p * r / (p + r)
    # This computes the harmonic mean of precision and recall.
    # F1 provides a balanced score when precision and recall trade off.

    metric_rows.append({
        "run_name": run_name,
        # This stores the run identifier so you can match metrics to hyperparameters.
        # It is essential for comparing results across the three trained models.

        "mAP50": map50,
        # This stores mAP50 as the accuracy proxy requested by the course.
        # Higher mAP50 generally indicates better localization and classification performance.

        "Precision": p,
        # This stores the precision value for the run.
        # Precision helps you assess how many detections are false alarms.

        "Recall": r,
        # This stores the recall value for the run.
        # Recall helps you assess how many real objects were missed by the model.

        "F1": f1,
        # This stores the computed F1 score for the run.
        # F1 summarizes the balance between precision and recall in one number.

        "weights": best_pt
        # This stores the weights path used for evaluation.
        # Including it supports reproducibility and auditability in your submission.
    })
    # This appends one completed row of metrics for the current run.
    # Collecting rows enables an easy Pandas table for the deliverable.

metrics_table = pd.DataFrame(metric_rows)
# This converts the metrics list into a formatted Pandas DataFrame.
# The table is required as one of your activity deliverables.

metrics_table
# This displays the metrics table in the notebook output.
# A visible table satisfies the deliverable for metrics reporting.

### Saving to Gdrive

In [ ]:
# =========================
# CELL 11 — Copy run artifacts to Drive + keep deliverables in one place
# =========================

LOCAL_RUNS_DIR = "/content/runs"
# This sets the local folder where Ultralytics wrote training outputs.
# We copy it to Drive so your results persist after the Colab runtime ends.

DEST_RUNS_DIR = os.path.join(DRIVE_RUNS_DIR, "detect_runs")
# This sets the destination folder inside your Drive course folder.
# Keeping a dedicated detect_runs folder makes sharing and submission easier.

shutil.copytree(LOCAL_RUNS_DIR, DEST_RUNS_DIR, dirs_exist_ok=True)
# This copies all run folders and artifacts (weights, plots, confusion matrices) into Drive.
# Copying ensures you can generate cloud links without rerunning training.

print("✅ Runs copied to Drive:", DEST_RUNS_DIR)
# This confirms that artifacts were saved to your Drive folder successfully.
# You can now share your Drive folder link as the “cloud link” deliverable.

hp_table_path = os.path.join(COURSE_ROOT, "hyperparameters_table.csv")
# This defines where the hyperparameter table CSV will be saved in Drive.
# Saving the CSV supports your report writing and submission packaging.

metrics_table_path = os.path.join(COURSE_ROOT, "metrics_table.csv")
# This defines where the metrics table CSV will be saved in Drive.
# Saving the CSV supports your PDF report and makes results easy to reference.

hp_table.to_csv(hp_table_path, index=False)
# This exports the hyperparameter table to Drive as a CSV file.
# It provides a clean artifact you can include in your submission or report appendix.

metrics_table.to_csv(metrics_table_path, index=False)
# This exports the metrics table to Drive as a CSV file.
# It provides a clean artifact you can include in your submission or report appendix.

print("✅ Saved:", hp_table_path)
# This prints the saved path so you can locate it in Drive quickly.
# It also confirms that the export completed successfully.

print("✅ Saved:", metrics_table_path)
# This prints the saved path so you can locate it in Drive quickly.
# It also confirms that the export completed successfully.

## Printouts

In [ ]:
# =========================
# CELL 12 — Print “final epoch results” from results.csv (for screenshot / text output deliverable)
# =========================

for cfg in RUNS:
    # This loops through each run to locate its results.csv file.
    # The results.csv contains the per-epoch metrics and losses.

    results_csv = os.path.join(TRAIN_PROJECT, "detect", cfg["run_name"], "results.csv")
    # This constructs the path to the run’s results.csv file.
    # The file is typically generated by Ultralytics during training.

    print("\n===== FINAL EPOCH SUMMARY FOR:", cfg["run_name"], "=====")
    # This prints a header so your outputs are clearly separated by run.
    # Clear separation makes it easier to screenshot as a deliverable.

    if os.path.exists(results_csv):
        # This checks whether results.csv exists before reading it.
        # It prevents runtime errors if the file path differs or training failed.

        df = pd.read_csv(results_csv)
        # This loads the results.csv into a DataFrame for easy inspection.
        # The DataFrame structure makes it straightforward to get the last epoch line.

        display(df.tail(1))
        # This displays the last row, which corresponds to the final epoch.
        # You can screenshot this output to satisfy the “final epoch results” deliverable.

    else:
        # This branch handles missing results.csv files.
        # It flags which run needs attention if artifacts were not generated.
        print("❌ results.csv not found at:", results_csv)
# This ends the loop after checking all runs.
# At this point you should have 3 final-epoch outputs visible in the notebook.

## Interpretation

In [ ]:
# =========================
# CELL 13 — Short confusion-matrix interpretation helper (for your discussion)
# =========================

classes = data_cfg.get("names", None)
# This extracts class names from data.yaml if available.
# Knowing class names helps you interpret diagonal true positives and off-diagonal confusions.

print("Class names from data.yaml:", classes)
# This prints the class list so you know which rows/columns correspond to which class.
# You will reference these names in your report discussion.

print("\nDiscussion checklist:")
# This prints a checklist to guide your narrative in the PDF report.
# The checklist ensures you cover true positives, false positives, and class struggles.

print("- True Positives = diagonal cells (correct class predictions).")
# This states the key rule for reading confusion matrices.
# It helps you identify which class is being detected correctly.

print("- False Positives often show as background→class or other-class→class confusions.")
# This explains a common pattern in detection confusion matrices.
# It connects directly to your smoke/fire false alarm analysis requirement.

print("- Identify which class has the worst confusion with background or other classes.")
# This tells you exactly what to look for when deciding which class struggled most.
# Your report must connect that observation to Precision/Recall/F1 differences.

### References

Ultralytics. (2026, January 14). *Ultralytics YOLO26*. Ultralytics YOLO Docs. https://docs.ultralytics.com/models/yolo26/

Ultralytics. (n.d.). *ultralytics/ultralytics* [GitHub repository]. GitHub. https://github.com/ultralytics/ultralytics

Roboflow. (2026, January 13). *Using the Python SDK*. Roboflow Docs. https://docs.roboflow.com/developer/python-sdk/using-the-python-sdk

The pandas development team. (n.d.). *Citing pandas*. pandas.pydata.org. https://pandas.pydata.org/about/citing.html

Paszke, A., Gross, S., Massa, F., Lerer, A., Bradbury, J., Chanan, G., Killeen, T., Lin, Z., Gimelshein, N., Antiga, L., Desmaison, A., Köpf, A., Yang, E., DeVito, Z., Raison, M., Tejani, A., Chilamkurthy, S., Steiner, B., Fang, L., … Chintala, S. (2019). *PyTorch: An imperative style, high-performance deep learning library* (arXiv:1912.01703). arXiv. https://doi.org/10.48550/arXiv.1912.01703